# LifeMate (code baseline): bước 2, LLM

Train LoRA cho **Qwen3-4B-Instruct-2507** trên quỹ đạo gold của `splits/train2_ids.txt`, chấm trên `splits/dev2_ids.txt` (300 mẫu, khuôn câu không có trong train2), rồi tạo bài nộp public bằng ASR đã fine-tune ở `colab_baseline.ipynb`.

Chuẩn bị:
1. **Runtime → GPU A100.**
2. Trên Drive đã có `MyDrive/zalo_lifemate/train+public_test.zip` và `MyDrive/zalo_lifemate/models/phowhisper_large_ft_fp16/` (từ notebook ASR), secret `GH_TOKEN` như trước.

Bấm **Chạy tất cả**. Dự kiến khoảng 2,5 giờ: cài đặt 10 phút, chép lời public 20 phút, train khoảng 1 giờ, chấm dev2 20 phút, chạy public 30 phút.
Mỗi bước lưu kết quả lên Drive; chạy lại lần sau sẽ bỏ qua bước nào đã có (ASR public, adapter LoRA).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 0. Code + data + thư viện

In [ ]:
import os
from google.colab import drive, userdata
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/zalo_lifemate"
REPO = "/content/zalo_lifemate"
token = userdata.get("GH_TOKEN")
if not os.path.isdir(REPO):
    !git clone -q https://{token}@github.com/FirstKhoi/zalo_lifemate.git {REPO}
    !git -C {REPO} remote set-url origin https://github.com/FirstKhoi/zalo_lifemate.git
%cd {REPO}
!git pull -q https://{token}@github.com/FirstKhoi/zalo_lifemate.git main && git log --oneline -1
os.environ["PYTHONPATH"] = f"{REPO}/src"
ASR_DRIVE = f"{DRIVE}/models/phowhisper_large_ft_fp16"
assert os.path.isfile(f"{DRIVE}/train+public_test.zip"), f"chưa có {DRIVE}/train+public_test.zip trên Drive"
assert os.path.isdir(ASR_DRIVE), f"chưa có {ASR_DRIVE}: chạy colab_baseline.ipynb trước"
if not os.path.isdir("train+public_test"):
    !unzip -q "{DRIVE}/train+public_test.zip" -d .
!mkdir -p out && ls train+public_test

In [ ]:
# vllm kéo theo torch của nó; torchao bản của Colab làm peft lỗi import nên gỡ đi
!pip install -q -r requirements.txt vllm trl peft datasets && pip uninstall -y -q torchao
!python -m pytest -q tests 2>&1 | tail -2
# nếu cell sau báo lỗi lạ (SyntaxError ở dòng !, lỗi import): Runtime → Restart session rồi chạy lại từ đầu

## 1. Chép lời public_test, không dùng domain prompt
Trên dev, bỏ prompt cho WER 8,65% so với 8,91% khi có prompt. Cache cũ trên Drive được làm với prompt nên làm lại một lần (khoảng 20 phút, không in gì tới khi xong).

In [ ]:
PUB_ASR = f"{DRIVE}/asr/asr_cache_public_test_noprompt.json"
if os.path.isfile(PUB_ASR):
    print("Đã có trên Drive, bỏ qua:", PUB_ASR)
else:
    # tên file mới: precompute_asr.py dùng tiếp cache cũ nếu --out đã tồn tại
    !python scripts/precompute_asr.py --data ./train+public_test --split public_test --backend hf --model {ASR_DRIVE} --gpus 1 --no-metrics --out out/asr_cache_public_test_noprompt.json 2>&1 | tee out/asr_public.log | grep --line-buffered -E "wrote|ERROR|Error|Traceback"
    !grep -A 40 Traceback out/asr_public.log | tail -45
    !mkdir -p "{DRIVE}/asr" && cp out/asr_cache_public_test_noprompt.json "{PUB_ASR}" && ls -la "{DRIVE}/asr"

## 2. Dữ liệu SFT + train LoRA
1245 quỹ đạo, mỗi quỹ đạo khoảng 5.300 token (dài nhất 8.800). Loss chỉ tính trên lượt assistant (`--assistant-only`). Thanh tiến trình hiện thời gian còn lại; cứ 10 bước in một dòng `{'loss': ...}`.

In [ ]:
!python scripts/build_sft.py --data ./train+public_test --split train --ids-file splits/train2_ids.txt --out out/sft_train.jsonl

In [ ]:
LORA = "models/qwen3-4b-sft"        # train_lora.py lưu adapter ở đây và bản đã merge ở LORA + "-merged"
MERGED = LORA + "-merged"           # = llm_model trong configs/baseline.json
LORA_DRIVE = f"{DRIVE}/lora_baseline"
if os.path.isfile(f"{LORA_DRIVE}/adapter_model.safetensors"):
    print("Đã có adapter trên Drive, bỏ qua train:", LORA_DRIVE)
else:
    # 1 GPU x grad-accum 16 = cùng batch hiệu dụng với README (4 GPU x 4)
    !python scripts/train_lora.py --train out/sft_train.jsonl --out {LORA} --epochs 2 --lr 1e-4 --r 16 --max-len 10240 --dtype bfloat16 --grad-accum 16 --assistant-only
    # chỉ adapter (khoảng 130MB) lên Drive; bản merge 8GB dựng lại từ adapter khi cần
    !mkdir -p "{LORA_DRIVE}" && cp {LORA}/adapter_config.json {LORA}/adapter_model.safetensors "{LORA_DRIVE}/" && ls -la "{LORA_DRIVE}"
if not os.path.isdir(MERGED):
    !python scripts/merge_lora.py --adapter "{LORA_DRIVE}" --out {MERGED}
!ls {MERGED}

## 3. Chấm trên dev2
- **oracle** (300 mẫu, lời thoại chuẩn): đo riêng phần LLM, dùng để so các bản model với nhau.
- **ASR** (64 mẫu `splits/dev2_asr_clean_ids.txt`): chỉ những mẫu mà ASR chưa từng train trên audio của chúng. 236 mẫu còn lại của dev2 có audio nằm trong dữ liệu train ASR nên chép lời gần như hoàn hảo, không phản ánh public.

In [ ]:
!python scripts/run_dev.py --data ./train+public_test --split train --ids-file splits/dev2_ids.txt --oracle --config configs/baseline.json --out-dir out/dev2_oracle 2>&1 | tee out/dev2_oracle.log | grep --line-buffered -E "^\[dev\]|ERROR|Error|Traceback"
!grep -A 40 Traceback out/dev2_oracle.log | tail -45
!python scripts/run_dev.py --data ./train+public_test --split train --ids-file splits/dev2_asr_clean_ids.txt --asr-cache "{DRIVE}/asr/asr_dev_noprompt.json" --config configs/baseline.json --out-dir out/dev2_asr_clean 2>&1 | tee out/dev2_asr_clean.log | grep --line-buffered -E "^\[dev\]|ERROR|Error|Traceback"
!grep -A 40 Traceback out/dev2_asr_clean.log | tail -45
import json
for run in ("dev2_oracle", "dev2_asr_clean"):
    s = json.load(open(f"out/{run}/summary.json"))
    print(f"{run}: final={s['final']:.2f} n={s['n']} perfect={s['perfect']} zero={s['zero']}",
          {k: round(v["score"], 1) for k, v in s["by_category"].items()}, "p95=%.1fs" % s["latency"]["p95"])

In [ ]:
!python scripts/error_buckets.py --run out/dev2_oracle --top 6 --per 2 | head -80
!mkdir -p "{DRIVE}/runs" && cp -r out/dev2_oracle out/dev2_asr_clean "{DRIVE}/runs/" && ls "{DRIVE}/runs"

## 4. Bài nộp public_test

In [ ]:
!python scripts/infer.py --data ./train+public_test --split public_test --config configs/baseline.json --asr-cache "{PUB_ASR}" --out out/public/submission.jsonl --no-resume 2>&1 | tee out/public.log | grep --line-buffered -E "^\[infer\] ([0-9]+/|done)|ERROR|Error|Traceback"
!grep -A 40 Traceback out/public.log | tail -45
!python scripts/check_submission.py --data ./train+public_test --split public_test --sub out/public/submission.jsonl

In [ ]:
!cp out/public/submission.jsonl "{DRIVE}/runs/submission_sft.jsonl" && ls -la "{DRIVE}/runs"
from google.colab import files
files.download("out/public/submission.jsonl")

## Xong
1. Nộp `submission.jsonl` vừa tải ở mục **Submission** trên portal (tối đa 5 lượt/ngày).
2. Gửi lại: 2 dòng `dev2_oracle: final=...` / `dev2_asr_clean: final=...` ở mục 3, và điểm public.
3. Tải thư mục `MyDrive/zalo_lifemate/runs/dev2_oracle/` và `dev2_asr_clean/` về `runs/` trên máy để phân tích lỗi.